<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week04-rag/Nugget020_Confidence_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-genai

In [21]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [22]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [30]:
evidence = {
    "dormant_accounts": 95,
    "privileged_dormant": 30,
    "jml_avg_hours": 36,
    "source_health": "GREEN"
}

In [31]:
def calculate_confidence(evidence):

    score = 0

    if evidence["dormant_accounts"] > 50:
        score += 1

    if evidence["privileged_dormant"] > 10:
        score += 1

    if evidence["jml_avg_hours"] > 24:
        score += 1

    if evidence["source_health"] == "GREEN":
        score += 1

    return score

In [32]:
print(calculate_confidence(evidence))

4


In [33]:
def confidence_label(score):

    if score >= 4:
        return "HIGH"

    elif score >= 2:
        return "MEDIUM"

    return "LOW"

In [34]:
score = calculate_confidence(evidence)

print(confidence_label(score))

HIGH


In [41]:
prompt = f"""
You are a senior IAM analyst.

Evidence:

{evidence}

Confidence Level:

{confidence_label(score)}

Generate:

1. Findings
2. Risk Assessment
3. Why confidence is {confidence_label(score)}
"""

In [42]:
print(callGPT(prompt).text)

As a Senior IAM Analyst, here is my assessment based on the provided evidence:

---

### 1. Findings

Based on the provided data, the following key observations have been identified:

*   **Significant Dormant Account Population:** There are 95 identified dormant accounts within the environment. These accounts represent potential attack vectors if compromised.
*   **High-Risk Privileged Dormant Accounts:** A critical subset of the dormant accounts, 30 are categorized as privileged. These accounts possess elevated access rights and pose a significantly higher risk if exploited.
*   **Sub-optimal Joiner-Mover-Leaver (JML) Process Efficiency:** The average time for JML processes is 36 hours. While this impacts joiner and mover efficiency, it is particularly concerning for the "Leaver" process, indicating a delay in de-provisioning access for former employees or contractors.
*   **Healthy Identity Data Sources:** The `source_health` is reported as 'GREEN', indicating that authoritative ide

In [43]:
prompt = f"""
Evidence:

{evidence}

Evaluate confidence.

Output JSON:

{{
 "confidence":"HIGH|MEDIUM|LOW",
 "reason":"..."
}}
"""

In [44]:
print(callGPT(prompt).text)

{
 "confidence": "LOW",
 "reason": "The evidence reveals 30 'privileged dormant' accounts, which represents a significant security vulnerability. Privileged accounts, even when dormant, pose a high risk if compromised due to their elevated access. Additionally, 95 'dormant_accounts' also contribute to the attack surface. While the 'source_health' being 'GREEN' indicates that the data source is reliable, the data itself points to substantial security weaknesses. The meaning and impact of 'jml_avg_hours: 36' are ambiguous without further context but do not sufficiently offset the clear risks from the dormant privileged accounts."
}
